# Reproduce the FIR hardware simulation

Run the recovered VHDL baseline and the streaming specialization on a **CPU runtime**. This is digital hardware simulation, with no model training, GPU, FPGA board, or Vivado requirement.

The shared verification runner has passed locally: 22 Python tests and 5,272 checked RTL outputs. This notebook has been syntax-checked, but has **not been executed in hosted Google Colab**. Your execution provides a separate reproducibility result; package versions may differ from the local Windows run and are recorded in the exported metadata.

1. In Colab, choose a standard Python CPU runtime.
2. Run the cells in order and upload `fir_filter_source.zip` when requested.
3. Download `results_for_review.zip` from the final cell and return that archive for review.

The source package includes code, tests, documentation, and source checksums. SHA-256 checks detect corruption; they do not establish the authorship of an unfamiliar archive. Use the archive supplied with this project.


In [1]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
import csv
import hashlib
import io
import json
import re
import stat
import subprocess
import sys
import zipfile

from IPython.display import JSON, Markdown, display
from google.colab import files

print("Python:", sys.version)
print("Runtime: CPU; no training or board access is performed.")


Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Runtime: CPU; no training or board access is performed.


## Install the VHDL simulator

Install the GHDL package supplied by the current Colab Ubuntu environment. The actual version is saved by the runner. Core verification uses the Python standard library.


In [2]:
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "ghdl"], check=True)
subprocess.run(["ghdl", "--version"], check=True)


CompletedProcess(args=['ghdl', '--version'], returncode=0)

## Upload the source archive

Select the generated `fir_filter_source.zip`. A new timestamped directory is created for each extraction. Existing project or result directories are not reused.


In [3]:
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one source ZIP archive.")
archive_name, source_archive = next(iter(uploaded.items()))
if not archive_name.lower().endswith(".zip"):
    raise ValueError("The source must be a ZIP archive.")
print("Uploaded archive SHA-256:", hashlib.sha256(source_archive).hexdigest())


Saving fir_filter_source.zip to fir_filter_source.zip
Uploaded archive SHA-256: 3263b77c5ea37b9a637d3452e162fa2ca6738d659a4426d4e6cf131e1d78451f


In [4]:
ROOT_FILES = {
    "README.md", "PROJECT_AUDIT.md", ".gitignore", "run_simulation.ps1",
    "requirements-figures.txt", "source_checksums.json",
}
SOURCE_DIRS = {
    "rtl", "tb", "scripts", "tests", "docs", "notebooks", "provenance", ".github",
}
TEXT_SUFFIXES = {".vhd", ".vhdl", ".py", ".md", ".json", ".ipynb", ".tcl",
                 ".ps1", ".txt", ".yml", ".yaml", ".xdc"}

with zipfile.ZipFile(io.BytesIO(source_archive)) as archive:
    entries = archive.infolist()
    if not 1 <= len(entries) <= 1000:
        raise ValueError("Unexpected source archive file count.")
    if sum(entry.file_size for entry in entries) > 32 * 1024 * 1024:
        raise ValueError("Source archive exceeds the expected size limit.")
    payloads = {}
    for entry in entries:
        name = entry.filename
        parts = name.split("/")
        path = PurePosixPath(name)
        mode = (entry.external_attr >> 16) & 0xFFFF
        if (entry.is_dir() or path.is_absolute() or "\\" in name or ":" in name
                or any(part in {"", ".", "..", "__pycache__"} for part in parts)
                or name in payloads or stat.S_ISLNK(mode)
                or entry.file_size > 8 * 1024 * 1024):
            raise ValueError(f"Unexpected archive member: {name!r}")
        permitted = (len(parts) == 1 and name in ROOT_FILES) or (
            len(parts) > 1 and parts[0] in SOURCE_DIRS
            and path.suffix.lower() in TEXT_SUFFIXES)
        permitted = permitted or (len(parts) == 3 and parts[:2] == ["results", "summary"]
                                   and path.suffix.lower() in {".png", ".json", ".csv", ".txt", ".md"})
        if not permitted:
            raise ValueError(f"Member outside the source allowlist: {name!r}")
        payload = archive.read(entry)
        if path.suffix.lower() != ".png":
            payload.decode("utf-8")
        payloads[name] = payload

manifest = json.loads(payloads["source_checksums.json"])
if manifest.get("schema_version") != 1 or manifest.get("algorithm") != "sha256":
    raise ValueError("Unrecognized source checksum manifest.")
expected_hashes = manifest["files"]
if not isinstance(expected_hashes, dict):
    raise ValueError("Invalid checksum file map.")
if set(expected_hashes) != set(payloads) - {"source_checksums.json"}:
    raise ValueError("Archive files do not exactly match the checksum manifest.")
for name, digest in expected_hashes.items():
    if (not isinstance(digest, str) or re.fullmatch(r"[0-9a-f]{64}", digest) is None
            or hashlib.sha256(payloads[name]).hexdigest() != digest):
        raise ValueError(f"Source checksum mismatch: {name}")

required = {
    "scripts/run_verification.py", "scripts/fir_reference.py",
    "rtl/fir_filter.vhd", "rtl/fir_filter_streaming.vhd",
    "tb/tb_fir_filter_original.vhd", "tb/tb_fir_verification.vhd",
    "tests/test_reference.py", "tests/test_result_validation.py",
}
if not required.issubset(payloads):
    raise ValueError(f"Missing required sources: {sorted(required - payloads.keys())}")

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
project_dir = Path("/content") / f"fir_filter_run_{stamp}"
project_dir.mkdir(parents=False, exist_ok=False)
for name, payload in payloads.items():
    target = project_dir.joinpath(*PurePosixPath(name).parts)
    target.parent.mkdir(parents=True, exist_ok=True)
    with target.open("xb") as handle:
        handle.write(payload)
print(f"Verified and extracted {len(expected_hashes)} source files into {project_dir}")


Verified and extracted 34 source files into /content/fir_filter_run_20261021T045946_656298Z


## Execute the complete verification

The runner executes Python unit tests, compiles the actual VHDL, runs the recovered original stimulus, and runs assertion-based tests against independent arithmetic references. It checks directed inputs, randomized traffic, reset recovery, handshake behavior, and an identical 512-sample benchmark for both architectures.

A successful run exports measured cycle counts and per-output comparisons. It does not measure FPGA resource use, physical maximum clock frequency, or power. If a cell fails, preserve the error and partial logs; do not interpret partial output as a passing run.


In [5]:
run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
results_dir = project_dir / "results" / f"colab_{run_stamp}"
subprocess.run(
    [sys.executable, "scripts/run_verification.py", "--output", str(results_dir)],
    cwd=project_dir,
    check=True,
)
metrics = json.loads((results_dir / "metrics.json").read_text(encoding="utf-8"))
if metrics.get("status") != "PASS":
    raise RuntimeError("The runner did not produce a passing result.")


## Inspect the results

Latency and block duration use elapsed rising-edge intervals. At the same clock frequency, the expected initiation intervals are five cycles for the baseline and one for the streaming core. The benchmark records actual simulation observations, and the runner checks them against the reference arithmetic and timing contract.


In [6]:
display(JSON(metrics))
with (results_dir / "benchmark.csv").open(newline="", encoding="utf-8") as handle:
    benchmark = list(csv.DictReader(handle))
headers = list(benchmark[0])
table = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
table += ["| " + " | ".join(row[column] for column in headers) + " |" for row in benchmark]
display(Markdown("\n".join(table)))
display(Markdown((results_dir / "evaluation_report.md").read_text(encoding="utf-8")))
print((results_dir / "python_tests.txt").read_text(encoding="utf-8"))


<IPython.core.display.JSON object>

| architecture | samples | first_accept_cycle | last_output_cycle | elapsed_cycles | initiation_interval_cycles |
| --- | --- | --- | --- | --- | --- |
| baseline | 512 | 2 | 2561 | 2559 | 5 |
| streaming | 512 | 2 | 514 | 512 | 1 |

# Verification results

Both VHDL designs passed assertion-based behavioral simulation.

Checked outputs: {'streaming': 3652, 'baseline': 1620}. Arithmetic mismatches: 0.

| Architecture | Samples | Elapsed cycles | Initiation interval |
|---|---:|---:|---:|
| baseline | 512 | 2559 | 5 |
| streaming | 512 | 512 | 1 |

At a common clock, the measured steady-state acceptance rate increases 5x.
This is a cycle-level simulation comparison, not measured FPGA Fmax, area, or power.
The fixed-coefficient streaming extension does not implement the report's abstract shared-multiplier resource model.
Vivado synthesis, physical timing closure, and board validation remain separate work.


test_empty_and_invalid (test_reference.ReferenceTests.test_empty_and_invalid) ... ok
test_full_scale_step (test_reference.ReferenceTests.test_full_scale_step) ... ok
test_impulse (test_reference.ReferenceTests.test_impulse) ... ok
test_independent_convolution (test_reference.ReferenceTests.test_independent_convolution) ... ok
test_new_run_clears_history (test_reference.ReferenceTests.test_new_run_clears_history) ... ok
test_report_sequence (test_reference.ReferenceTests.test_report_sequence) ... ok
test_arithmetic_mismatch_rejected (test_result_validation.ResultValidationTests.test_arithmetic_mismatch_rejected) ... ok
test_benchmark_index_corruption_rejected (test_result_validation.ResultValidationTests.test_benchmark_index_corruption_rejected) ... ok
test_consistent_fixture (test_result_validation.ResultValidationTests.test_consistent_fixture) ... ok
test_disagreement_between_architectures_rejected (test_result_validation.ResultValidationTests.test_disagreement_between_architectures_r

## Optional figures

Set `MAKE_FIGURES = True` to generate a comparison chart and the normalized frequency response. This uses the separate plotting dependency and is not required for verification. Figures are written after the review archive is finalized and are intentionally not added to that archive. The RTL retains DC gain six; normalization applies only to the frequency-response display.


In [7]:
MAKE_FIGURES = False
if MAKE_FIGURES:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", "requirements-figures.txt"],
        cwd=project_dir, check=True,
    )
    subprocess.run(
        [sys.executable, "scripts/plot_results.py", str(results_dir)],
        cwd=project_dir, check=True,
    )
    from IPython.display import Image
    for figure in sorted((results_dir / "figures").glob("*.png")):
        display(Image(filename=str(figure)))


## Download the evidence package

The archive contains `metrics.json`, `environment.json`, the CSV comparisons, simulator logs, Python test results, VCD waveforms, source snapshots, an evaluation report, and SHA-256 checksums. The next cell checks its manifest before downloading.

Return **`results_for_review.zip`**. No training results or model checkpoints are needed. If execution failed, return the error text and available logs instead of claiming a successful run.


In [8]:
review_archive = results_dir / "results_for_review.zip"
with zipfile.ZipFile(review_archive) as archive:
    names = archive.namelist()
    if len(names) != len(set(names)):
        raise ValueError("Duplicate entries in the review archive.")
    result_hashes = json.loads(archive.read("checksums.json"))
    if set(names) != set(result_hashes) | {"checksums.json"}:
        raise ValueError("Review archive does not match its manifest.")
    for name, digest in result_hashes.items():
        if hashlib.sha256(archive.read(name)).hexdigest() != digest:
            raise ValueError(f"Result checksum mismatch: {name}")
print("Verified review archive SHA-256:", hashlib.sha256(review_archive.read_bytes()).hexdigest())
print("Download:", review_archive)
files.download(str(review_archive))


Verified review archive SHA-256: e4e7bd7271f6dc18e3da6589fcba7955a8e24361a4f103d04b69422c64172a0d
Download: /content/fir_filter_run_20261021T045946_656298Z/results/colab_20261021T045946_666106Z/results_for_review.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>